In [ ]:
# experiment from original umap notebook but now with larger data set (MNIST)
# source for code: https://github.com/lmcinnes/umap_paper_notebooks/blob/master/CrossVal%20KNN%20Classifier%20-%20MNIST.ipynb

In [16]:
import numpy as np
import pandas as pd
import sklearn.neighbors
import sklearn.model_selection
import sklearn.datasets
import sklearn.manifold
import sklearn.decomposition

In [17]:
def score_embedding(data, labels, k=10, n_neighbors=5, random_state=42):
    model = sklearn.neighbors.KNeighborsClassifier(n_neighbors=n_neighbors)
    cv = sklearn.model_selection.StratifiedKFold(n_splits=k)
    # only change to original code was removing random_state = random_state, as newer scikit-learn versions require shuffle=True when random_state is set
    # without shuffling, the folds follow the original data order, which reproduces the standard deviations of the paper (same goes for smaller dataset code)
    return sklearn.model_selection.cross_val_score(model, data, labels, cv=cv)

In [18]:
def compare_embeddings(embeddings, n_neighbors=5, k=10, random_state=42):
    cross_val_scores = {}
    for alg in embeddings.keys():
        cross_val_scores[alg] = score_embedding(embeddings[alg][['x', 'y']].values, 
                                                embeddings[alg]['labels'].values,
                                                n_neighbors=n_neighbors,
                                                k=k,
                                                random_state=random_state)
    return pd.DataFrame(cross_val_scores).assign(n_neighbors=n_neighbors)

In [20]:
import numpy as np
import pandas as pd
import umap
import sklearn.manifold
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
from tensorflow.keras.datasets import mnist
# from sklearn.datasets import fetch_mldata
# the import above was used in the original code but has been completely removed from scikit-learn several versions ago
# code taken from https://github.com/lmcinnes/umap_paper_notebooks/blob/master/UMAP%20MNIST.ipynb
(X_train, y_train), (X_test, y_test) = mnist.load_data()
data = X_train.reshape(-1, 784).astype('float32')
labels = y_train.astype(int)
fit = umap.UMAP(n_neighbors=15, random_state=42, metric='euclidean')
u = fit.fit_transform(data)
output = pd.DataFrame(u, columns=('x','y'))
output['labels'] = labels
output.to_csv('embedding_mnist_umap1.csv')

fit_tsne = TSNE(n_jobs=1, random_state=42) # used TSNE from sklearn instead of MulticoreTSNE (as used by the authors) because MulticoreTSNE didnt want to install
u_tsne = fit_tsne.fit_transform(data)
output = pd.DataFrame(u_tsne, columns=('x','y'))
output['labels'] = labels
output.to_csv('embedding_mnist_tsne1.csv')

pca = PCA(n_components=2)
u_pca = pca.fit_transform(data)
output = pd.DataFrame(u_pca, columns=('x','y'))
output['labels'] = labels
output.to_csv('embedding_mnist_pca.csv')

/home/reinoutvrielink/.local/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [21]:
mnist = {}
mnist['tsne'] = pd.read_csv('embedding_mnist_tsne1.csv', index_col=0)
mnist['umap'] = pd.read_csv('embedding_mnist_umap1.csv', index_col=0)
# Skipped creating the LargeVis embeddings myself because the original implementation 
# relies on a legacy C++ codebase that (according to Claude) requires manual compilation and external 
# libraries (like GSL), which frequently fail on modern systems
# That is why we use the pre computed embeddings by the authors
mnist['lvis'] = pd.read_csv('embedding_mnist_largevis1.csv', index_col=0)
mnist['leig'] = pd.read_csv('embedding_mnist_eigenmaps1.csv', index_col=0) # creating the embeddings myself would have taken at least 11 hours (that is how long it took the authors and they have better hardware then me)
mnist['pca'] = pd.read_csv('embedding_mnist_pca.csv', index_col=0)

In [22]:
df = pd.concat(compare_embeddings(mnist, n_neighbors=n, k=20) for n in (100,200,400,800,1600,3200))

In [23]:
df.groupby('n_neighbors').mean()

,tsne,umap,lvis,leig,pca
n_neighbors,,,,,
100,0.966850,0.966633,0.962086,0.668200,0.460150
200,0.965817,0.966267,0.961500,0.667386,0.464400
400,0.965333,0.966067,0.961371,0.664171,0.467167
800,0.964783,0.965350,0.961086,0.659929,0.466417
1600,0.963283,0.964050,0.947243,0.651157,0.463183
3200,0.959967,0.961233,0.920014,0.639129,0.455350


In [24]:
df.groupby('n_neighbors').std() * 2

,tsne,umap,lvis,leig,pca
n_neighbors,,,,,
100,0.010142,0.010594,0.014869,0.016062,0.020037
200,0.010887,0.010496,0.014822,0.016122,0.021828
400,0.011012,0.010511,0.014996,0.016165,0.022526
800,0.011419,0.010533,0.015452,0.016915,0.021663
1600,0.011689,0.010401,0.015310,0.014787,0.021243
3200,0.010812,0.010795,0.017526,0.017354,0.020977
